In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from utils import *
import prompt

In [ ]:
representetive_gene_list = (repository_root() / "examples/merfish/representative_genes.txt").read_text().splitlines()


# data

In [ ]:
config = load_config("configs/config_oneshot_merfish.yaml")
config.data_name = "MERFISH_27"
config.refresh_paths()
name_truth = config.name_truth

In [ ]:
# --- Load data ---
data_path = str(dataset_file("merfish", config.data_name))
adata = sc.read_h5ad(data_path) 
# rename the column of cell_class to cell_type
adata.obs.rename(columns={'cell_class': 'cell_type'}, inplace=True)

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
adata.obs = adata.obs.join(pos_data)

# rename cell types
celltype_rename = {
    'Astrocyte' : 'Astrocyte',
 'Endothelial 1': 'Endothelial',
 'OD Mature 2': 'Mature oligodendrocytes',
 'Inhibitory': 'Inhibitory',
 'OD Immature 1': 'Immature oligodendrocytes',
 'Excitatory': 'Excitatory',
 'Endothelial 3': 'Endothelial',
 'Microglia': 'Microglia',
 'OD Mature 1': 'Mature oligodendrocytes',
 'Pericytes': 'Pericytes',
 'OD Mature 4': 'Mature oligodendrocytes',
 'Endothelial 2': 'Endothelial',
 'OD Mature 3': 'Mature oligodendrocytes',
 'OD Immature 2': 'Immature oligodendrocytes',
 'Ependymal': 'Ependymal'
}
adata.obs['cell_type'] = adata.obs['cell_type'].map(celltype_rename)
celltype_data = adata.obs[['cell_type']]

sc.pp.filter_genes(adata, min_cells=5)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.scale(adata)


# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
# add diagonal to the adj_matrix
adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))



In [ ]:


# get all the top genes
top_genes = representetive_gene_list

# --- Calculate neighbor genes ---
neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                              index=adata.obs_names, 
                              columns=top_genes)


## sample data

In [ ]:
# 设定随机种子
seed = 42  # 你可以根据需要修改这个值

# 定义分割比例 p (比如 0.7 表示 70% 数据用于训练，30% 数据用于测试)
p = config.prototype_p

# 分割数据集为训练集和测试集
train_neighbor_normalized_df, val_neighbor_normalized_df = train_test_split(neighbor_normalized_df, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[config.name_truth]
                                                           )


train_neighbor_normalized_df_genes, val_neighbor_normalized_df_genes = train_test_split(neighbor_normalized_df_genes, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[config.name_truth]
                                                           )


In [ ]:
# check sample distribution
adata.obs[config.name_truth].loc[train_neighbor_normalized_df.index].value_counts()

# prompt

In [ ]:
unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

# the full cell type names are already in the data
unique_celltypes = adata.obs['cell_type'].unique()
cell_names_mapping = {celltype: celltype for _, celltype in enumerate(unique_celltypes)}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping

config.cell_names = neighbor_normalized_df.columns
config.gene_names = top_genes


In [ ]:
# calculate prototype
train_neighbor_df = train_neighbor_normalized_df.join(train_neighbor_normalized_df_genes).copy()
one_shot_df = pd.concat([adata.obs[config.name_truth].loc[train_neighbor_df.index], train_neighbor_df], axis=1).groupby(config.name_truth, observed=False).mean()
print(one_shot_df.index)

# generate Comparison-based Prompt
config.oneshot_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)

In [ ]:
x = [i for i in range(len(adata)) if adata.obs[config.name_truth].iloc[i] == "MPA"][30:31]
print(config.oneshot_prompt + prompt.oneshot_celltype_geneorder(neighbor_normalized_df, neighbor_normalized_df_genes, x, config))


# GPT

## generate json

In [ ]:
generate_json_end2end(val_neighbor_normalized_df, config, prompt.oneshot_celltype_geneorder, batch_size = 3000, n_rows = 1, df_extra=val_neighbor_normalized_df_genes)


## submit

In [ ]:
# submit_end2end.py
# nohup python -u -m src.submit_end2end configs/config_BZ9_zeroshot.yaml > BZ9_zeroshot.out 2>&1 &
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_oneshot_merfish.yaml MERFISH_27 _rep1 > outs/MERFISH_27_oneshot_rep1.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 2
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")


In [ ]:
gpt_results_df.columns = ['oneshot_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)
gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")

In [ ]:
gpt_results_df = gpt_results_df[[0]]
gpt_results_df.columns = ['oneshot_gpt4o_mini']

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.oneshot_gpt4o_mini.value_counts()

In [ ]:
# # manually retrieve batch output
# output_file_name = f"{output_path}/response_BZ5_1_{use_full_name}_{with_self_type}_{with_region_name}_{Graph_type}_{with_negatives}_{with_CoT}_{with_count_numbers}{config.replicate}.txt"
# batch_id = "batch_66f62020d11c81909424c1423da11a70"
# file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
# # Open the file in write mode and save the string
# with open(output_file_name, 'w') as file:
#     file.write(file_response.text)

# Gemini

In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)


In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, 
                                                val_neighbor_normalized_df, config, 
                                                prompt.oneshot_celltype_geneorder, n_rows=1, 
                                                df_extra=val_neighbor_normalized_df_genes,
                                                column_name="oneshot_gemini")

with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
gemini_results_df.index = gemini_results_df.index.astype(str)

In [ ]:
# which index number of gemini_results_df.index == '1'
# [i for i in range(len(gemini_results_df)) if gemini_results_df.index[i] == '1']



In [ ]:
gemini_results_df.index.difference(val_neighbor_normalized_df.index)

In [ ]:
gemini_results_df.index = gemini_results_df.index.str.replace("id_", "")

In [ ]:
gemini_results_df['oneshot_gemini'].value_counts()

# plot

In [ ]:
val_adata = adata[val_neighbor_normalized_df.index,].copy()
# val_adata.obs = val_adata.obs.join(gemini_results_df)
val_adata.obs = val_adata.obs.join(gpt_results_df)
# val_adata.obs['oneshot_gemini'] = val_adata.obs['oneshot_gemini'].fillna("unknown")
val_adata.obs['oneshot_gpt4o_mini'] = val_adata.obs['oneshot_gpt4o_mini'].fillna("unknown")
# sc.pl.scatter(val_adata, x="x", y="y", color="oneshot_gemini", title =  f"oneshot_gemini")
sc.pl.scatter(val_adata, x="x", y="y", color="oneshot_gpt4o_mini", title =  f"oneshot_gpt4o_mini")

In [ ]:
# print(adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['oneshot_gemini']))
print(adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['oneshot_gpt4o_mini']))


In [ ]:
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

# test

## load test data BZ9 BZ14 
Prototype in prompt is based on training data

In [ ]:
config = load_config("configs/config_BZ9_oneshotTest.yaml")

domain_mapping = {1 : "Layer 1", 2 : "Layer 2/3", 3 : "Layer 5", 4 : "Layer 6"}
cell_names_mapping = {'Astro': 'Astrocytes',
 'Endo': 'Endothelial cells',
 'L5-1': 'Layer 5 pyramidal neuron subtype 1',
 'Lhx6': 'Lhx6-expressing interneurons',
 'NPY': 'Neuropeptide Y-expressing interneurons',
 'Oligo': 'Oligodendrocytes',
 'Reln': 'Reelin-expressing cells',
 'SST': 'Somatostatin-expressing interneurons',
 'Smc': 'Smooth muscle cells',
 'VIP': 'Vasoactive intestinal peptide-expressing interneurons',
 'eL2/3': 'Excitatory neuron layer 2/3',
 'eL5-2': 'Excitatory neuron layer 5 subtype 2',
 'eL5-3': 'Excitatory neuron layer 5 subtype 3',
 'eL6-1': 'Excitatory neuron layer 6 subtype 1',
 'eL6-2': 'Excitatory neuron layer 6 subtype 2'}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping
config.pos_data = pos_data
config.celltype_data = celltype_data

# prompt is based on training data
config.oneshot_prompt = prompt.CP_celltype(one_shot_df, config)

In [ ]:
# --- Load data ---
data_path = str(dataset_dir("starmap", config.data_name))
x_data_name = "data.csv"  
index_col = 0
adata = sc.read_csv(f"{data_path}/{x_data_name}", first_column_names=True)
celltype_data = pd.read_csv(f"{data_path}/celltype.csv", index_col=index_col)  # Assuming first column is index
celltype_data.columns = ["cell_type"] 
pos_data = pd.read_csv(f"{data_path}/pos.csv", index_col=index_col)
pos_data.columns = ['x', 'y']
domain_data = pd.read_csv(f"{data_path}/domain.csv", index_col=index_col)
domain_data.columns = [config.name_truth]
adata.obs = adata.obs.join([celltype_data, pos_data, domain_data])

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

pos_data = adata.obs[['x', 'y']]
celltype_data = adata.obs[['cell_type']]
domain_data = adata.obs[[config.name_truth]]

# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
# add diagonal to the adj_matrix
adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))


## test GPT

In [ ]:
generate_json_end2end(neighbor_normalized_df, config, prompt.oneshot_celltype_geneorder, batch_size = 5000, n_rows = 1, df_extra=neighbor_normalized_df_genes)

In [ ]:
# submit_end2end.py
# nohup python -u -m src.submit_end2end configs/config_BZ9_zeroshot.yaml > BZ9_zeroshot.out 2>&1 &
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_BZ9_oneshotTest.yaml > BZ9_oneshotTest.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ").replace("‘", "'").replace("’", "'")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['oneshot_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.value_counts()

## test Gemini

In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)


In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, 
                                                neighbor_normalized_df, config, 
                                                prompt.oneshot_celltype_geneorder, n_rows=1, 
                                                df_extra=neighbor_normalized_df_genes,
                                                column_name="oneshot_gemini")

with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

gemini_results_df.index = gemini_results_df.index.astype(str)



In [ ]:
gemini_results_df.index.difference(adata.obs.index)


In [ ]:

gemini_results_df.value_counts()

## plot

In [ ]:

adata.obs = adata.obs.join(gemini_results_df)
adata.obs = adata.obs.join(gpt_results_df)
adata.obs['oneshot_gemini'] = adata.obs['oneshot_gemini'].fillna("unknown")
adata.obs['oneshot_gpt4o_mini'] = adata.obs['oneshot_gpt4o_mini'].fillna("unknown")
sc.pl.scatter(adata, x="x", y="y", color="oneshot_gemini", title =  f"oneshot_gemini")
sc.pl.scatter(adata, x="x", y="y", color="oneshot_gpt4o_mini", title =  f"oneshot_gpt4o_mini")

In [ ]:
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['oneshot_gemini']))
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['oneshot_gpt4o_mini']))


## save results

In [ ]:
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
